# Exact Best-Response Feasibility Frontier

This notebook identifies where dense exact exploitability stops being operationally practical.

It first estimates dense dimensions without allocating them. Selected candidates are then evaluated in isolated subprocesses using random `512x512` neural policies. A random policy is sufficient for this benchmark because policy quality does not materially change dense tensor shape, neural compilation cost, or exact-BR tree size.

A failed or OOM-killed evaluator does not take down the notebook kernel.

In [ ]:
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.policies.neural import NeuralPolicy
from liars_poker.serialization import save_policy

print('Python:', sys.version)
print('CPU count:', os.cpu_count())

In [ ]:
def symmetric_hand_count(spec):
    # Number of rank multisets of hand_size, with at most suits copies per rank.
    counts = [0] * (spec.hand_size + 1)
    counts[0] = 1
    for _ in range(spec.ranks):
        updated = [0] * (spec.hand_size + 1)
        for used, ways in enumerate(counts):
            for copies in range(min(spec.suits, spec.hand_size - used) + 1):
                updated[used + copies] += ways
        counts = updated
    return counts[spec.hand_size]


def hand_count(spec):
    if spec.suit_symmetry and spec.suits > 1:
        return symmetric_hand_count(spec)
    return math.comb(spec.ranks * spec.suits, spec.hand_size)


def cgroup_memory():
    root = Path('/sys/fs/cgroup')
    maximum_path = root / 'memory.max'
    current_path = root / 'memory.current'
    if not maximum_path.exists():
        return None, None
    maximum_text = maximum_path.read_text().strip()
    maximum = None if maximum_text == 'max' else int(maximum_text)
    current = int(current_path.read_text()) if current_path.exists() else None
    return maximum, current


def human_bytes(value):
    value = float(value)
    for unit in ('B', 'KiB', 'MiB', 'GiB', 'TiB', 'PiB'):
        if abs(value) < 1024.0:
            return f'{value:.2f} {unit}'
        value /= 1024.0
    return f'{value:.2f} EiB'


memory_limit, memory_current = cgroup_memory()
print('cgroup memory limit:', human_bytes(memory_limit) if memory_limit else 'unlimited/unknown')
print('cgroup memory current:', human_bytes(memory_current) if memory_current else 'unknown')

## Candidate ladder

The ladder separates public-history growth, private-hand growth, and loss of suit symmetry. The final two rows demonstrate why adding all claim families or combining large rank counts with physical hands leaves the dense regime.

In [ ]:
CANDIDATES = {
    'reference r4 h2 hp2pt ss': GameSpec(4, 4, 2, ('RankHigh', 'Pair', 'TwoPair', 'Trips'), True),
    'more private r4 h3 hp2pt ss': GameSpec(4, 4, 3, ('RankHigh', 'Pair', 'TwoPair', 'Trips'), True),
    'rank growth r6 h3 hpt ss': GameSpec(6, 4, 3, ('RankHigh', 'Pair', 'Trips'), True),
    'rank growth r7 h3 hpt ss': GameSpec(7, 4, 3, ('RankHigh', 'Pair', 'Trips'), True),
    'rank growth r8 h3 hpt ss': GameSpec(8, 4, 3, ('RankHigh', 'Pair', 'Trips'), True),
    'physical hands r4 h2 hpt nss': GameSpec(4, 4, 2, ('RankHigh', 'Pair', 'Trips'), False),
    'physical hands r4 h3 hpt nss': GameSpec(4, 4, 3, ('RankHigh', 'Pair', 'Trips'), False),
    'combined r6 h3 hpt nss': GameSpec(6, 4, 3, ('RankHigh', 'Pair', 'Trips'), False),
    'all current claims r4 h2 ss': GameSpec(
        4, 4, 2,
        ('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
        True,
    ),
}

# The observed peak on r4_s4_h2_hp2pt_ss was roughly seven times the raw
# dense arrays. Eight is a conservative empirical planning multiplier, not a
# proof or allocation guarantee.
EMPIRICAL_PEAK_MULTIPLIER = 8.0

dimension_rows = []
for label, candidate in CANDIDATES.items():
    k = len(rules_for_spec(candidate).claims)
    histories = 1 << k
    hands = hand_count(candidate)
    actions = k + 1
    strategy_bytes = histories * hands * actions * 4
    reach_bytes = 2 * histories * hands * 4
    structural_bytes = histories * actions + histories * 6
    raw_dense_bytes = strategy_bytes + reach_bytes + structural_bytes
    dimension_rows.append({
        'label': label,
        'short spec': candidate.to_short_str(),
        'claims k': k,
        'histories 2^k': histories,
        'private hands N': hands,
        'actions A': actions,
        'strategy GiB': strategy_bytes / 2**30,
        'reach GiB': reach_bytes / 2**30,
        'raw dense GiB': raw_dense_bytes / 2**30,
        'planning peak GiB': EMPIRICAL_PEAK_MULTIPLIER * raw_dense_bytes / 2**30,
    })

dimensions_df = pd.DataFrame(dimension_rows).set_index('label')
display(dimensions_df.style.format({
    'strategy GiB': '{:.3f}',
    'reach GiB': '{:.3f}',
    'raw dense GiB': '{:.3f}',
    'planning peak GiB': '{:.3f}',
}))

## Isolated probes

The default probe set is intentionally moderate. Add a candidate only after inspecting the dimension table. Set `ALLOW_OVER_BUDGET=True` only when deliberately testing failure behavior.

In [ ]:
PROBE_LABELS = [
    'reference r4 h2 hp2pt ss',
    'more private r4 h3 hp2pt ss',
    'rank growth r6 h3 hpt ss',
    'physical hands r4 h3 hpt nss',
]
MAX_MEMORY_FRACTION = 0.60
TIMEOUT_HOURS_PER_PROBE = 6
ALLOW_OVER_BUDGET = False
COMPILE_BATCH_SIZE = 65_536
EVALUATOR_CPU_THREADS = 16

RUN_DIR = REPO_ROOT / 'artifacts' / 'exact_br_frontier'
RUN_DIR.mkdir(parents=True, exist_ok=True)

WORKER_CODE = r'''
import json
import sys
from pathlib import Path
repo_root = Path(sys.argv[1])
policy_dir = Path(sys.argv[2])
iteration = int(sys.argv[3])
compile_batch_size = int(sys.argv[4])
sys.path.insert(0, str(repo_root))
from liars_poker.eval.deep_cfr_async import evaluate_saved_neural_policy
result = evaluate_saved_neural_policy(
    policy_dir,
    iteration=iteration,
    label='frontier_probe',
    compile_batch_size=compile_batch_size,
)
print(json.dumps(result))
'''


def process_memory(pid):
    path = Path(f'/proc/{pid}/status')
    if not path.exists():
        return None
    values = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.startswith(('VmRSS:', 'VmHWM:')):
            key, value, _ = line.split()
            values[key.rstrip(':')] = int(value) * 1024
    return values


def run_probe(label, spec, estimated_peak_bytes):
    if (
        memory_limit is not None
        and estimated_peak_bytes > MAX_MEMORY_FRACTION * memory_limit
        and not ALLOW_OVER_BUDGET
    ):
        return {
            'label': label,
            'status': 'skipped by memory budget',
            'planning peak GiB': estimated_peak_bytes / 2**30,
        }

    policy_dir = RUN_DIR / spec.to_short_str() / 'policy'
    if not (policy_dir / 'metadata.json').exists():
        save_policy(NeuralPolicy(spec, hidden_sizes=(512, 512)), str(policy_dir))
    result_path = policy_dir / 'result.json'
    if result_path.exists():
        result_path.unlink()

    environment = os.environ.copy()
    for variable in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
        environment[variable] = str(EVALUATOR_CPU_THREADS)

    process = subprocess.Popen(
        [
            sys.executable, '-u', '-c', WORKER_CODE,
            str(REPO_ROOT), str(policy_dir), '0', str(COMPILE_BATCH_SIZE),
        ],
        cwd=REPO_ROOT,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    start = time.perf_counter()
    deadline = start + TIMEOUT_HOURS_PER_PROBE * 3600
    peak_rss = 0
    while process.poll() is None:
        memory = process_memory(process.pid)
        if memory:
            peak_rss = max(peak_rss, memory.get('VmRSS', 0), memory.get('VmHWM', 0))
        if time.perf_counter() >= deadline:
            process.kill()
            stdout, stderr = process.communicate()
            return {
                'label': label,
                'status': 'timeout',
                'wall s': time.perf_counter() - start,
                'peak sampled RSS GiB': peak_rss / 2**30,
                'stderr': stderr[-2000:],
            }
        time.sleep(1.0)

    stdout, stderr = process.communicate()
    result = json.loads(result_path.read_text(encoding='utf-8')) if result_path.exists() else {}
    result.update({
        'label': label,
        'status': 'complete' if process.returncode == 0 else 'failed',
        'return code': process.returncode,
        'wall s': time.perf_counter() - start,
        'peak sampled RSS GiB': peak_rss / 2**30,
        'planning peak GiB': estimated_peak_bytes / 2**30,
        'stderr': stderr[-2000:],
    })
    return result

In [ ]:
RUN_PROBES = True
probe_results = []

if RUN_PROBES:
    for label in PROBE_LABELS:
        candidate = CANDIDATES[label]
        planning_peak_bytes = (
            dimensions_df.loc[label, 'planning peak GiB'] * 2**30
        )
        print(f'\n=== {label} ===')
        result = run_probe(label, candidate, planning_peak_bytes)
        probe_results.append(result)
        print(result.get('status'), 'wall min=', result.get('wall s', 0) / 60.0)

probe_df = pd.DataFrame(probe_results).set_index('label')
display(probe_df)

In [ ]:
completed = probe_df[probe_df['status'] == 'complete'].copy()
if not completed.empty:
    display(completed[[
        'dense_compile_s', 'exact_br_s', 'evaluation_s',
        'peak sampled RSS GiB', 'planning peak GiB',
    ]].style.format(precision=3))

    joined = completed.join(dimensions_df[['claims k', 'private hands N', 'raw dense GiB']])
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    axes[0].scatter(joined['raw dense GiB'], joined['dense_compile_s'])
    axes[1].scatter(joined['raw dense GiB'], joined['exact_br_s'])
    axes[2].scatter(joined['raw dense GiB'], joined['peak sampled RSS GiB'])
    for label, row in joined.iterrows():
        for ax, y in zip(axes, ('dense_compile_s', 'exact_br_s', 'peak sampled RSS GiB')):
            ax.annotate(label, (row['raw dense GiB'], row[y]), fontsize=8)
    for ax, title, ylabel in [
        (axes[0], 'Dense compilation scaling', 'Compilation seconds'),
        (axes[1], 'Exact BR scaling', 'Exact BR seconds'),
        (axes[2], 'Evaluator memory scaling', 'Peak sampled RSS GiB'),
    ]:
        ax.set(title=title, xlabel='Raw dense-array GiB', ylabel=ylabel)
        ax.grid(True, alpha=0.3)
    fig.tight_layout();

## Interpreting the frontier

A spec remains in the practical exact regime only if one isolated learned-average evaluation fits comfortably in RAM and completes within the operational time budget. Once a candidate is skipped, times out, or approaches the memory threshold, larger specs should use exact evaluation only after evaluator redesign—or switch to calibrated approximate evaluation.